# 第 6 周练习：游戏发行商提交分类（Publisher Triage）

## 练习目标

帮助游戏发行商给涌入的推介（pitch）排优先级：把每条标成 **High / Medium / Low**。

## 流程

1. **数据集** — 内置（pitch, priority）样例；可选经 **OpenRouter** 用 LLM 再生成一批
2. **前沿基线（Frontier baseline）** — 在测试集上做零样本分类，报告 **准确率（accuracy）**（这是要被微调击败的数字）
3. **导出 JSONL** — 把训练/验证数据写到 `jsonl/`，便于别处做开源或 OpenAI 微调
4. **评估** — 再报一遍基线准确率，并展示若干条预测样例

## 和本课第 6 周的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| 数据准备与划分 | train / val / test 固定种子切分 |
| 前沿模型零样本基线 | OpenRouter + `gpt-4o-mini` |
| 微调数据格式 | Chat messages → JSONL |

## 怎么跑

1. `.env` 里设置 `OPENROUTER_API_KEY`
2. 从上到下运行；生成更多样例、跑基线评估的格子会真实调用 API


In [57]:
# ========== 导入与 OpenRouter 客户端 ==========

# 标准库：环境变量、JSON、随机划分、正则清 markdown 围栏
import os
import json
import random
import re
# 从 .env 加载密钥
from dotenv import load_dotenv
# OpenAI SDK：指向 OpenRouter 的兼容端点
from openai import OpenAI

# override=True：用 .env 覆盖已有环境变量
load_dotenv(override=True)
# OpenRouter 的 OpenAI 兼容 Base URL（保持原样）
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
# 前沿小模型 id（保持原样）
FRONTIER_MODEL = "openai/gpt-4o-mini"

# 读 OpenRouter API Key
api_key = os.getenv("OPENROUTER_API_KEY")
# 创建客户端：自定义 base_url + key
client = OpenAI(base_url=OPENROUTER_BASE_URL, api_key=api_key)


## 数据集：游戏提交的推介与优先级

合成的（pitch, priority）样例：

- **High** = 契合度高、钩子清晰、接近完工
- **Medium** = 扎实但偏通用，或仍偏早期
- **Low** = 非常早期、表述模糊，或与发行方向不匹配


In [58]:
# ========== 标签集合 + 合成推介数据集 ==========
# PRIORITIES：允许的优先级标签（必须与 prompt / 准确率比对用的英文一致）
# SUBMISSIONS：每条含 pitch（推介文案）与 priority（High/Medium/Low）
# 下列字典字段与字符串保持英文原样，供零样本分类与 JSONL 导出

PRIORITIES = ("High", "Medium", "Low")

SUBMISSIONS = [
    {"pitch": "Roguelike deckbuilder, 80% complete, Steam page live. Strong wishlist. Team of 3, shipped 2 games. Looking for QA and marketing support for Q2 launch.", "priority": "High"},
    {"pitch": "Narrative puzzle game, vertical slice done, targeting Switch and PC. Art style similar to Gris. Need funding for 8 months to release.", "priority": "High"},
    {"pitch": "Co-op roguelike for 2–4 players, alpha in 3 months. Clear scope, proven genre. Seeking advance and porting help for console.", "priority": "High"},
    {"pitch": "Pixel art metroidvania, demo on Steam. 50k wishlists. Solo dev, 12 months to launch. Need QA and localization.", "priority": "High"},
    {"pitch": "Tactics RPG with job system, beta in 2 months. Full design doc and GDD. Team of 5, one shipped title. PC first, then Switch.", "priority": "High"},
    {"pitch": "Idle clicker with narrative twist, soft launch data positive. Mobile + Steam. Looking for publisher for global release.", "priority": "High"},
    {"pitch": "Horror walking sim, vertical slice and trailer ready. Strong mood and story. 6 months to beta. PC and console.", "priority": "High"},
    {"pitch": "Local multiplayer party game, 4 players, alpha playable. Ideal for Switch. Team of 2, previous jam experience.", "priority": "High"},
    {"pitch": "Turn-based strategy with hex map, 70% complete. Similar to Into the Breach. Need marketing and store presence.", "priority": "High"},
    {"pitch": "Narrative adventure, branching dialogue, demo on itch. Strong writing sample. 9 months to release. PC only.", "priority": "High"},
    {"pitch": "We are making a cool RPG game. It will have lots of quests and fighting. Still in early development. No demo yet.", "priority": "Low"},
    {"pitch": "My friend and I have an idea for a multiplayer shooter. We have some concept art. Looking for funding to start development.", "priority": "Low"},
    {"pitch": "Open-world survival game with crafting. Very early stage. We think it could be like Minecraft meets Rust. No playable build.", "priority": "Low"},
    {"pitch": "Mobile puzzle game. We have a prototype but no clear hook. Would like publisher support for everything.", "priority": "Low"},
    {"pitch": "An MMO with unique combat. Still in pre-production. Need full funding and team. No vertical slice.", "priority": "Low"},
    {"pitch": "We want to make the next big battle royale. Concept only. No technical demo. Looking for full budget.", "priority": "Low"},
    {"pitch": "Casual mobile game, idea stage. No build. Need publisher to help with design and development.", "priority": "Low"},
    {"pitch": "RPG with 100 hours of content. Solo dev, 6 months in, no playable demo. Would need long timeline.", "priority": "Low"},
    {"pitch": "Social deduction game for 10+ players. Early prototype, unproven on market. No clear platform.", "priority": "Low"},
    {"pitch": "Retro platformer, very early. We have a short level. Looking for advance to work full-time for 2 years.", "priority": "Low"},
    {"pitch": "2D platformer with metroidvania elements. Demo available, decent controls. Genre is crowded. No clear differentiator yet.", "priority": "Medium"},
    {"pitch": "Puzzle game inspired by Portal. Prototype works but art is placeholder. Need 6 months and funding to polish.", "priority": "Medium"},
    {"pitch": "Tower defense with RPG progression. Alpha in progress. Solid mechanics, art style not final. PC first.", "priority": "Medium"},
    {"pitch": "Narrative game with choices. Writing is strong, gameplay is light. Vertical slice in 4 months. Small scope.", "priority": "Medium"},
    {"pitch": "Roguelike shooter, early alpha. Fun core loop, needs content and balance. Team of 2, first commercial game.", "priority": "Medium"},
    {"pitch": "Co-op puzzle adventure. Concept is clear, build is 30% done. Would need 12 months and QA support.", "priority": "Medium"},
    {"pitch": "Deckbuilder with unique theme. Prototype playable, balance rough. Looking for feedback and possible deal.", "priority": "Medium"},
    {"pitch": "Horror puzzle game. Mood is good, puzzles need work. No release date. Solo dev, part-time.", "priority": "Medium"},
    {"pitch": "Action RPG, alpha stage. Combat feels good, story and scope still growing. Need 18 months.", "priority": "Medium"},
    {"pitch": "Party game for 2–4 players. Mini-games are fun, presentation is early. Targeting Switch, 9 months to beta.", "priority": "Medium"},
    {"pitch": "Souls-like 2D, hard difficulty. Niche but dedicated. Demo on Steam, modest wishlist. Need marketing.", "priority": "High"},
    {"pitch": "Management sim, city-builder hybrid. Beta in 4 months. Data from beta testers positive. PC + console.", "priority": "High"},
    {"pitch": "Story-driven RPG, no combat. Demo and trailer ready. Strong writing, 6 months to launch. PC only.", "priority": "High"},
    {"pitch": "Rhythm game with custom tracks. Community tools ready. Early access in 2 months. Niche but engaged.", "priority": "High"},
    {"pitch": "Tactical stealth game, vertical slice done. Clear vision, small team. Need QA and porting for console.", "priority": "High"},
    {"pitch": "Colony sim with survival elements. Alpha playable, roadmap clear. 12 months to 1.0. PC first.", "priority": "High"},
    {"pitch": "We have a dream to make an open-world game. No team, no demo, no design doc. Just an idea.", "priority": "Low"},
    {"pitch": "Multiplayer only game, no single player. Very early prototype. Unclear monetization.", "priority": "Low"},
    {"pitch": "Sequel to our unreleased first game. First game had no traction. Looking for publisher for both.", "priority": "Low"},
    {"pitch": "Fighting game with 20 characters. Two people, 1 year in. No animator. Scope is very large.", "priority": "Low"},
    {"pitch": "Educational game for kids. Noble goal but no gameplay hook. Early prototype.", "priority": "Low"},
    {"pitch": "Top-down shooter, functional prototype. Generic theme. Could be polished in 6 months with support.", "priority": "Medium"},
    {"pitch": "Visual novel with mini-games. Script 50% done. Art style consistent. Need 8 months and voice budget.", "priority": "Medium"},
    {"pitch": "Racing game with power-ups. Core mechanics work, content thin. First project for team of 3.", "priority": "Medium"},
    {"pitch": "Dungeon crawler, tile-based. Alpha available. Classic feel, needs UX pass and content.", "priority": "Medium"},
    {"pitch": "Endless runner with roguelike elements. Mobile + PC. Prototype is fun, monetization not decided.", "priority": "Medium"},
    {"pitch": "Automation game, factorio-like. Complex systems, alpha. Long development ahead, small team.", "priority": "Medium"},
    {"pitch": "Bullet hell shooter. Tight controls, content in progress. Niche audience. 6 months to early access.", "priority": "Medium"},
    {"pitch": "Mystery adventure, point-and-click. Story outlined, first chapter playable. Need funding for full game.", "priority": "Medium"},
    {"pitch": "Twin-stick shooter with upgrades. Alpha, 4 months to beta. Indie scope, clear mechanics.", "priority": "Medium"},
    {"pitch": "Card battler with narrative. Demo on itch. Unique theme. Need 10 months and localization.", "priority": "High"},
    {"pitch": "Platform fighter, local + online. Netcode in progress. Beta in 5 months. Dedicated genre audience.", "priority": "High"},
    {"pitch": "Farming sim with RPG. Co-op planned. Vertical slice in 2 months. Strong reference (Stardew).", "priority": "High"},
    {"pitch": "Narrative puzzle horror. Demo and trailer ready. Small scope, 6 months. PC + console.", "priority": "High"},
    {"pitch": "Tactics game with permadeath. Alpha, clear design. 9 months to release. Need QA and marketing.", "priority": "High"},
    {"pitch": "Roguelike with daily runs. Meta-progression designed. Beta in 3 months. Mobile + PC potential.", "priority": "High"},
    {"pitch": "Puzzle platformer, physics-based. Demo polished. 4 months to launch. Solo dev, one previous release.", "priority": "High"},
    {"pitch": "Co-op horror, 2 players. Vertical slice done. Strong atmosphere. Need 8 months and porting.", "priority": "High"},
    {"pitch": "Strategy game, turn-based, hex. Beta in 1 month. Mod support planned. PC focus.", "priority": "High"},
    {"pitch": "Action platformer with speedrun focus. Demo on Steam. Community interest. 5 months to 1.0.", "priority": "High"},
    {"pitch": "Idle/incremental with narrative. Soft launch data good. Small scope. Need publisher for store and marketing.", "priority": "High"},
    {"pitch": "A game where you do stuff. We have ideas. Maybe 2 years of development. No prototype.", "priority": "Low"},
    {"pitch": "Battle royale with a twist. Pre-alpha. Large scope. Need full funding and team expansion.", "priority": "Low"},
    {"pitch": "VR game, concept only. No headset build. Looking for VR publisher and budget.", "priority": "Low"},
    {"pitch": "Sequel to a game that didn't sell. We believe in the IP. Need advance to finish.", "priority": "Low"},
    {"pitch": "Multiplayer only, 50 players per match. Small team. No technical demo. Very ambitious.", "priority": "Low"},
    {"pitch": "RPG with procedural world. Huge scope. One programmer. No vertical slice after 1 year.", "priority": "Low"},
    {"pitch": "Sports game, no license. Early prototype. Niche. Unclear differentiator.", "priority": "Medium"},
    {"pitch": "Narrative RPG, text-heavy. Good writing sample. Playable chapter 1. Need 14 months.", "priority": "Medium"},
    {"pitch": "Roguelike with crafting. Systems in place, content needed. First game, small team.", "priority": "Medium"},
    {"pitch": "Puzzle game with story. Mechanics proven, narrative in progress. 7 months to beta.", "priority": "Medium"},
    {"pitch": "Tower defense + RPG. Alpha. Fun core loop, needs variety. PC and mobile possible.", "priority": "Medium"},
    {"pitch": "Exploration game, minimal UI. Mood over mechanics. Vertical slice in 5 months. Art-heavy.", "priority": "Medium"},
    {"pitch": "Shooter with roguelike runs. Alpha. Good feel, content pipeline unclear. 10 months estimate.", "priority": "Medium"},
    {"pitch": "Social deduction, 6–8 players. Prototype tested at events. Needs content and polish.", "priority": "Medium"},
    {"pitch": "Racing + combat. Cars and weapons. Alpha. Niche genre. 8 months to early access.", "priority": "Medium"},
    {"pitch": "Base-building strategy. Single player focus. Alpha in 2 months. Clear scope, small team.", "priority": "Medium"},
]


## 划分训练 / 验证 / 测试

用固定随机种子把数据集切成 **训练 70%**、**验证 15%**、**测试 15%**，保证每次划分可复现。


In [ ]:
# ========== 固定种子划分 train / val / test ==========

# 种子 42：保证每次 shuffle 结果一致，方便对比实验
random.seed(42)
# 浅拷贝列表，避免 shuffle 打乱原始 SUBMISSIONS 顺序语义（原逻辑如此）
shuffled = SUBMISSIONS.copy()
random.shuffle(shuffled)
n = len(shuffled)
# 70% 训练、15% 验证，剩余归测试
n_train, n_val = int(0.7 * n), int(0.15 * n)
train_data = shuffled[:n_train]
val_data = shuffled[n_train : n_train + n_val]
test_data = shuffled[n_train + n_val :]
print(f"Train {len(train_data)}, Val {len(val_data)}, Test {len(test_data)}")


## （可选）用 LLM 生成更多样例

通过前沿模型再生成若干（pitch, priority），扩充 `SUBMISSIONS`。需要有效的 OpenRouter Key。


In [ ]:
# ========== 可选数据增强：让 LLM 按三类优先级生成推介 ==========

def generate_triage_examples(n_per_priority: int = 2):
    """Generate (pitch, priority) examples via OpenRouter. Appends to SUBMISSIONS."""
    # prompt 必须保持英文：直接约束模型输出 JSON 数组格式
    prompt = f"""Generate {n_per_priority} short game submission pitches for each priority: High, Medium, Low.
High = strong fit, near-complete, clear hook. Medium = solid but generic or early. Low = very early, vague, or mismatched.
Reply with a JSON array of objects: [{{"pitch": "one sentence pitch", "priority": "High"}}, ...]
Only the JSON array, no other text."""
    # 调用前沿模型；max_tokens 给足 JSON 数组空间
    r = client.chat.completions.create(model=FRONTIER_MODEL, messages=[{"role": "user", "content": prompt}], max_tokens=800)
    raw = (r.choices[0].message.content or "").strip()
    # 去掉可能包着的 ```json ... ``` 围栏（正则保持原样）
    raw = re.sub(r"^```(?:json)?\\s*", "", raw).strip()
    raw = re.sub(r"\\s*```$", "", raw).strip()
    try:
        extra = json.loads(raw)
        for ex in extra:
            # 只接纳字段齐全且 priority 合法的对象
            if isinstance(ex, dict) and ex.get("priority") in PRIORITIES and ex.get("pitch"):
                SUBMISSIONS.append(ex)
        print(f"Added {len(extra)} generated examples. Total: {len(SUBMISSIONS)}")
    except Exception as e:
        # 解析失败时打印原因，不中断笔记本
        print(f"Generation failed: {e}")

# 默认会执行：每类生成 2 条（需要 API）；注释掉可跳过联网调用
generate_triage_examples(2)  # uncomment to run


## 前沿零样本基线

在测试集上用 OpenRouter 做零样本分类，并报告准确率——这是后续微调要对齐/超越的基准。


In [ ]:
# ========== 零样本分类器 + 测试集准确率 ==========

# 标签元组：与数据集、system prompt 中的英文一致
PRIORITIES = ("High", "Medium", "Low")

def predict_baseline(pitch: str) -> str:
    """Zero-shot classification via OpenRouter (frontier baseline)."""
    # system prompt 保持英文：约束模型只回一个标签词
    sys_prompt = (
        "You are a game publisher triaging submissions. "
        "Classify the following pitch as exactly one of: High, Medium, Low. "
        "Reply with only that one word, nothing else."
    )
    r = client.chat.completions.create(
        model=FRONTIER_MODEL,
        messages=[
            {"role": "system", "content": sys_prompt},
            {"role": "user", "content": pitch},
        ],
        max_tokens=10,
    )
    raw = (r.choices[0].message.content or "").strip()
    # 宽松匹配：回复里包含某个标签（大小写不敏感）就规范化返回
    for label in PRIORITIES:
        if label.lower() in raw.lower():
            return label
    # 匹配失败：退回原文，或默认 Medium
    return raw or "Medium"

def accuracy(predictor, data):
    # 预测标签与真值 priority 完全相等才算对
    correct = sum(1 for ex in data if predictor(ex["pitch"]) == ex["priority"])
    return correct / len(data) if data else 0.0

# 在测试集上跑基线（会多次调用 API）
baseline_acc = accuracy(predict_baseline, test_data)
print(f"=== Frontier baseline (OpenRouter zero-shot) ===")
print(f"Accuracy on test set: {baseline_acc:.1%}")
print("This is the number to beat with fine-tuning.")


## 准备微调用的对话数据

把每条样本转成 `user = pitch` / `assistant = priority` 的 messages，再写成 JSONL。


In [62]:
# ========== JSONL 工具：messages 格式 ↔ 文件 ==========

def messages_for(example):
    """One training example: user = pitch, assistant = priority label."""
    # Chat 微调常见两条消息：问推介、答标签
    return [
        {"role": "user", "content": example["pitch"]},
        {"role": "assistant", "content": example["priority"]},
    ]


def make_jsonl(items):
    # 每行一个 JSON：{"messages": [...]}
    lines = [json.dumps({"messages": messages_for(ex)}) for ex in items]
    return "\n".join(lines)


def write_jsonl(items, filepath):
    # 若路径含目录则先创建；目录名为空时用 "."
    os.makedirs(os.path.dirname(filepath) or ".", exist_ok=True)
    with open(filepath, "w", encoding="utf-8") as f:
        f.write(make_jsonl(items))


## 导出到 `jsonl/`

把训练与验证集写成 JSONL。`write_jsonl` 会按需创建目录（不必事先手动建好）。


In [ ]:
# ========== 写出 train / validation JSONL ==========

# 输出目录名（相对当前工作目录）
JSONL_DIR = "jsonl"
# 训练集 → fine_tune_train.jsonl
write_jsonl(train_data, f"{JSONL_DIR}/fine_tune_train.jsonl")
# 验证集 → fine_tune_validation.jsonl
write_jsonl(val_data, f"{JSONL_DIR}/fine_tune_validation.jsonl")
print(f"Wrote {len(train_data)} train, {len(val_data)} val to {JSONL_DIR}/")


## JSONL 的去向

`jsonl/` 里的文件可交给 **OpenAI 微调**（若你有密钥）或开源微调工具。本笔记本本身只通过 **OpenRouter** 跑零样本基线。


In [ ]:
# ========== 确认导出路径与条数 ==========
# JSONL 已写在 jsonl/；需要时可拿到 OpenAI 或开源微调流水线使用
print(f"Training data: {JSONL_DIR}/fine_tune_train.jsonl ({len(train_data)} examples), {JSONL_DIR}/fine_tune_validation.jsonl ({len(val_data)} examples)")


## 再评估：OpenRouter 基线准确率

用同一 `predict_baseline` 在测试集上重算准确率（会再次调用 API）。


In [ ]:
# ========== 准确率函数（再次定义）+ 打印基线 ==========

def accuracy(predictor, data):
    # 与前面相同的准确率定义，便于本格独立重跑
    correct = sum(1 for ex in data if predictor(ex["pitch"]) == ex["priority"])
    return correct / len(data) if data else 0.0


# 在测试集上评估前沿零样本
acc_baseline = accuracy(predict_baseline, test_data)
print(f"Frontier baseline (OpenRouter): {acc_baseline:.1%}")


## 样例预测

下面展示若干测试集推介的真值与模型预测，帮助直观感受发行商分诊质量。


In [ ]:
# ========== 抽查前 5 条测试预测 ==========
# 每条会再调一次 OpenRouter 基线分类器
for ex in test_data[:5]:
    pred = predict_baseline(ex["pitch"])
    # True = 数据集标签；Predicted = 模型输出
    print(f"True: {ex['priority']} | Predicted: {pred}")
    # 推介正文只印前 80 字符，避免刷屏
    print(f"  {ex['pitch'][:80]}...")
    print()
